# Notebook 03 — Preprocessing + Feature Engineering

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction using GraphSAGE

### Objective

This notebook creates the **single source of truth for preprocessing** that will later be reused by the GraphSAGE training workflow and the production FastAPI service.

Workflow:

```text
Raw bank-full.csv
        ↓
Leakage / prediction-time policy
        ↓
Train / Validation / Test split
        ↓
Feature engineering
        ↓
ColumnTransformer
        ↓
Fitted preprocessor
        ↓
Dense customer feature matrix
        ↓
Saved artifacts
```

### Critical rules

1. The raw CSV is never modified.
2. The target `y` is separated before preprocessing.
3. The train/validation/test split is stratified.
4. The preprocessing pipeline is fitted **only on training data**.
5. Validation and test data are transformed using the training-fitted preprocessor.
6. No test information is used to fit preprocessing or choose transformations.
7. `duration` is treated as prediction-time leakage-sensitive and is excluded from the default deployment feature set.
8. Timing-sensitive variables such as `pdays`, `previous`, `poutcome`, `campaign`, `contact`, and `month` are retained but explicitly documented for prediction-time review.
9. The exact fitted preprocessor is saved for production reuse.

### Why `duration` is excluded

Notebook 02 identified `duration` as the primary leakage-sensitive feature. A deployment system that predicts whether to target a customer should not depend on the duration of the current call when that duration is not yet known.

Therefore:

```text
duration → excluded from deployment/model features
```

This is a modeling-policy decision, not a claim that `duration` is statistically unimportant.


## 1. Project Paths

Expected input:

```text
Bank-Marketing-GNN/
└── data/
    └── raw/
        └── bank-full.csv
```

Expected outputs:

```text
Bank-Marketing-GNN/
├── data/
│   ├── interim/
│   └── processed/
│
└── artifacts/
    ├── models/
    │   └── preprocessor.joblib
    └── metadata/
        └── feature_schema.json
```


In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TARGET_COLUMN = "y"

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

# Prediction-time policy
EXCLUDE_DURATION = True

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

for directory in [
    INTERIM_DIR,
    PROCESSED_DIR,
    MODELS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
FEATURE_SCHEMA_PATH = METADATA_DIR / "feature_schema.json"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Preprocessor artifact:", PREPROCESSOR_PATH)
print("Feature schema:", FEATURE_SCHEMA_PATH)


# 2. Load Raw Dataset

The dataset is loaded with the verified semicolon delimiter.

No modifications are made to the raw file.


In [ ]:
# ============================================================
# 2. Load Dataset
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH, sep=";")

assert df.shape == (45211, 17), (
    f"Unexpected dataset shape: {df.shape}"
)
assert TARGET_COLUMN in df.columns

print("Raw dataset loaded.")
print("Shape:", df.shape)
display(df.head())


# 3. Raw Data Quality Gate

Before creating model splits, verify the assumptions established during Notebook 01 and Notebook 02.


In [ ]:
# ============================================================
# 3. Data Quality Checks
# ============================================================

missing_cells = int(df.isna().sum().sum())
duplicate_rows = int(df.duplicated().sum())

constant_columns = [
    column
    for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

numeric_columns_raw = df.select_dtypes(include=np.number).columns.tolist()

infinite_values = int(
    np.isinf(df[numeric_columns_raw].to_numpy()).sum()
) if numeric_columns_raw else 0

assert missing_cells == 0, f"Missing cells detected: {missing_cells}"
assert duplicate_rows == 0, f"Duplicate rows detected: {duplicate_rows}"
assert not constant_columns, f"Constant columns detected: {constant_columns}"
assert infinite_values == 0, f"Infinite values detected: {infinite_values}"

print("Data quality gate passed.")
print(f"Missing cells      : {missing_cells}")
print(f"Duplicate rows     : {duplicate_rows}")
print(f"Constant columns   : {constant_columns}")
print(f"Infinite values    : {infinite_values}")


# 4. Define Target and Prediction Features

The target is separated before any preprocessing.

The default feature policy is:

- Numerical variables are scaled.
- Categorical variables are one-hot encoded.
- `duration` is excluded because of prediction-time leakage concerns.
- Other timing-sensitive variables remain available but are documented for later deployment review.


In [ ]:
# ============================================================
# 4. Feature Policy
# ============================================================

all_feature_columns = [
    column for column in df.columns
    if column != TARGET_COLUMN
]

leakage_sensitive_features = [
    "duration"
]

timing_sensitive_features = [
    "pdays",
    "previous",
    "poutcome",
    "campaign",
    "contact",
    "month",
]

excluded_features = []

if EXCLUDE_DURATION and "duration" in all_feature_columns:
    excluded_features.append("duration")

model_feature_columns = [
    column for column in all_feature_columns
    if column not in excluded_features
]

X_raw = df[model_feature_columns].copy()
y_raw = df[TARGET_COLUMN].copy()

print("Target:", TARGET_COLUMN)
print("Total raw features:", len(all_feature_columns))
print("Excluded features:", excluded_features)
print("Model features:", len(model_feature_columns))

print("\nTiming-sensitive features retained for review:")
print([
    column
    for column in timing_sensitive_features
    if column in model_feature_columns
])


# 5. Minimal, Justified Feature Engineering

This project intentionally avoids aggressive feature engineering.

The main objective is to establish a reliable preprocessing contract that can be reproduced exactly in production.

Two bounded transformations are used:

- `balance_log1p = log1p(balance)` to reduce the influence of extreme balance values.
- `campaign_log1p = log1p(campaign)` to compress the right-skewed campaign-contact count.

The original columns are retained as well. These transformations are computed row-by-row and do not learn parameters from the full dataset.

No target-derived features are created.


In [ ]:
# ============================================================
# 5. Feature Engineering
# ============================================================

def engineer_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    data = dataframe.copy()

    if "balance" in data.columns:
        # Bank balance can contain negative values, so shift by the
        # dataset-level minimum only after splitting would be leakage-prone.
        # Instead use a sign-preserving log transform:
        # sign(x) * log1p(abs(x)).
        data["balance_signed_log1p"] = (
            np.sign(data["balance"]) *
            np.log1p(np.abs(data["balance"]))
        )

    if "campaign" in data.columns:
        data["campaign_log1p"] = np.log1p(
            np.clip(data["campaign"], a_min=0, a_max=None)
        )

    if "previous" in data.columns:
        data["previous_log1p"] = np.log1p(
            np.clip(data["previous"], a_min=0, a_max=None)
        )

    return data

X_engineered = engineer_features(X_raw)

engineered_features = [
    column
    for column in X_engineered.columns
    if column not in X_raw.columns
]

print("Engineered features:")
print(engineered_features)

display(X_engineered.head())


# 6. Stratified Train / Validation / Test Split

The split is created **before fitting the preprocessing pipeline**.

Final proportions:

- Train: 70%
- Validation: 15%
- Test: 15%

The split is stratified by the target so that class proportions remain comparable across partitions.


In [ ]:
# ============================================================
# 6. Stratified Split
# ============================================================

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_engineered,
    y_raw,
    test_size=TEST_SIZE,
    stratify=y_raw,
    random_state=RANDOM_STATE,
)

# Validation size is expressed relative to the remaining 85%.
validation_relative_size = VALIDATION_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_relative_size,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

print("Split sizes:")
print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)


In [ ]:
# ============================================================
# 7. Verify Split Class Distributions
# ============================================================

def class_distribution(series):
    return pd.DataFrame({
        "count": series.value_counts(),
        "percentage": series.value_counts(normalize=True).mul(100).round(3)
    })

print("Overall:")
display(class_distribution(y_raw))

print("Train:")
display(class_distribution(y_train))

print("Validation:")
display(class_distribution(y_val))

print("Test:")
display(class_distribution(y_test))


# 7. Leakage Check

At this point:

- The target is separated.
- The test set has not been used to fit anything.
- The validation set has not been used to fit anything.
- The preprocessing pipeline will be fitted only on `X_train`.

This is the critical boundary for preventing preprocessing leakage.


In [ ]:
# ============================================================
# 8. Explicit Leakage Checks
# ============================================================

assert TARGET_COLUMN not in X_train.columns
assert TARGET_COLUMN not in X_val.columns
assert TARGET_COLUMN not in X_test.columns

assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0

print("Split leakage checks passed.")
print("Train/validation/test indices are mutually exclusive.")


# 8. Define Preprocessing Pipeline

### Numerical preprocessing

`StandardScaler` is fitted only on the training numerical features.

### Categorical preprocessing

`OneHotEncoder(handle_unknown="ignore")` is fitted only on training categories.

`handle_unknown="ignore"` is important for production because a future customer can contain a category that was not observed during training.

The complete `ColumnTransformer` is saved as a single artifact.


In [ ]:
# ============================================================
# 9. Identify Feature Groups After Engineering
# ============================================================

numerical_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal input features:", len(X_train.columns))


In [ ]:
# ============================================================
# 10. Build ColumnTransformer
# ============================================================

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("ColumnTransformer created.")


# 9. Fit on Training Data Only

This is the most important preprocessing operation in the notebook.

The pipeline learns:

- numerical means and standard deviations
- categorical vocabulary

**only from the training set**.

Validation and test sets are transformed afterward.


In [ ]:
# ============================================================
# 11. Fit Preprocessor on TRAIN ONLY
# ============================================================

preprocessor.fit(X_train)

print("Preprocessor fitted on training data only.")


In [ ]:
# ============================================================
# 12. Transform Train / Validation / Test
# ============================================================

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_val_processed = np.asarray(X_val_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

print("Processed shapes:")
print("Train      :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test       :", X_test_processed.shape)


In [ ]:
# ============================================================
# 13. Verify Processed Data
# ============================================================

assert X_train_processed.ndim == 2
assert X_val_processed.ndim == 2
assert X_test_processed.ndim == 2

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_val_processed).all()
assert np.isfinite(X_test_processed).all()

print("Processed feature validation passed.")
print("Final transformed feature count:", X_train_processed.shape[1])


# 10. Inspect Generated Feature Names

The exact transformed feature names are extracted from the fitted preprocessor.

These names become part of the production feature schema.


In [ ]:
# ============================================================
# 14. Extract Feature Names
# ============================================================

try:
    transformed_feature_names = preprocessor.get_feature_names_out().tolist()
except AttributeError:
    transformed_feature_names = [
        f"feature_{i}"
        for i in range(X_train_processed.shape[1])
    ]

assert len(transformed_feature_names) == X_train_processed.shape[1]
assert len(set(transformed_feature_names)) == len(transformed_feature_names)

print("Transformed feature count:", len(transformed_feature_names))
display(
    pd.DataFrame({
        "index": range(len(transformed_feature_names)),
        "feature_name": transformed_feature_names
    }).head(50)
)


# 11. Inspect Training Statistics

Check that numerical standardization behaves as expected on the training set.

Because categorical one-hot features are also present, this is an inspection rather than a requirement that every transformed column have unit variance.


In [ ]:
# ============================================================
# 15. Training Matrix Statistics
# ============================================================

train_feature_means = X_train_processed.mean(axis=0)
train_feature_stds = X_train_processed.std(axis=0)

print("Training matrix:")
print("Minimum value:", float(X_train_processed.min()))
print("Maximum value:", float(X_train_processed.max()))
print("Mean absolute feature mean:", float(np.abs(train_feature_means).mean()))
print("Median feature std:", float(np.median(train_feature_stds)))


# 12. Save Split Data

The split raw/engineered customer records are saved with their original row indices.

This is useful for Notebook 04 because graph construction needs to preserve the correspondence between:

```text
customer row
     ↕
customer node
     ↕
target
     ↕
train/validation/test mask
```

The processed matrices are also saved as NumPy arrays.


In [ ]:
# ============================================================
# 16. Save Split Data
# ============================================================

def save_split(name, X_split, y_split):
    split_df = X_split.copy()
    split_df[TARGET_COLUMN] = y_split.values
    split_df.to_csv(
        INTERIM_DIR / f"{name}_engineered.csv",
        index=True
    )

    np.save(
        PROCESSED_DIR / f"X_{name}.npy",
        preprocessor.transform(X_split).astype(np.float32)
    )

    np.save(
        PROCESSED_DIR / f"y_{name}.npy",
        y_split.to_numpy()
    )

save_split("train", X_train, y_train)
save_split("validation", X_val, y_val)
save_split("test", X_test, y_test)

print("Split datasets and processed matrices saved.")


In [ ]:
# ============================================================
# 17. Save Processed Arrays from Verified Matrices
# ============================================================

np.save(
    PROCESSED_DIR / "X_train.npy",
    X_train_processed
)

np.save(
    PROCESSED_DIR / "X_validation.npy",
    X_val_processed
)

np.save(
    PROCESSED_DIR / "X_test.npy",
    X_test_processed
)

np.save(
    PROCESSED_DIR / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    PROCESSED_DIR / "y_validation.npy",
    y_val.to_numpy()
)

np.save(
    PROCESSED_DIR / "y_test.npy",
    y_test.to_numpy()
)

print("Verified processed arrays saved.")


# 13. Save the Exact Fitted Preprocessor

This artifact is critical.

The production backend must use this exact fitted object rather than rebuilding preprocessing manually.


In [ ]:
# ============================================================
# 18. Save Preprocessor
# ============================================================

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

assert PREPROCESSOR_PATH.exists()

print(f"Preprocessor saved to: {PREPROCESSOR_PATH}")
print(f"File size: {PREPROCESSOR_PATH.stat().st_size / 1024:.2f} KB")


# 14. Save Feature Schema

The schema records:

- raw feature columns
- excluded columns
- engineered features
- numerical/categorical input groups
- transformed feature names
- target
- split proportions
- prediction-time policy
- leakage-sensitive variables

This metadata will later help the backend validate raw customer input.


In [ ]:
# ============================================================
# 19. Save Feature Schema
# ============================================================

feature_schema = {
    "target_column": TARGET_COLUMN,

    "dataset": {
        "rows": int(df.shape[0]),
        "raw_columns": df.columns.tolist()
    },

    "feature_policy": {
        "all_raw_features": all_feature_columns,
        "excluded_features": excluded_features,
        "model_input_features_before_encoding": X_train.columns.tolist(),
        "engineered_features": engineered_features,
        "leakage_sensitive_features": leakage_sensitive_features,
        "timing_sensitive_features_for_review": timing_sensitive_features,
        "duration_excluded_by_default": bool(EXCLUDE_DURATION),
    },

    "preprocessing": {
        "numerical_features": numerical_features,
        "categorical_features": categorical_features,
        "numeric_transform": "StandardScaler",
        "categorical_transform": "OneHotEncoder",
        "handle_unknown": "ignore",
        "remainder": "drop"
    },

    "transformed_features": {
        "count": len(transformed_feature_names),
        "names": transformed_feature_names
    },

    "split": {
        "train_fraction": round(len(X_train) / len(df), 6),
        "validation_fraction": round(len(X_val) / len(df), 6),
        "test_fraction": round(len(X_test) / len(df), 6),
        "random_state": RANDOM_STATE,
        "stratified": True
    }
}

with open(FEATURE_SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, indent=2)

assert FEATURE_SCHEMA_PATH.exists()

print(f"Feature schema saved to: {FEATURE_SCHEMA_PATH}")


# 15. Reload Test — Production Preprocessor Integrity

Before leaving this notebook, reload the saved preprocessor from disk.

This verifies that the artifact can be serialized and restored without changing its transformation behavior.


In [ ]:
# ============================================================
# 20. Reload Preprocessor and Compare Transformations
# ============================================================

loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

X_test_reloaded = loaded_preprocessor.transform(X_test)
X_test_reloaded = np.asarray(X_test_reloaded, dtype=np.float32)

np.testing.assert_allclose(
    X_test_processed,
    X_test_reloaded,
    rtol=1e-6,
    atol=1e-6
)

print("Reloaded preprocessor produces identical test transformations.")


# 16. Verify Saved Artifacts

The final gate checks that every required Notebook 03 artifact exists and is loadable.


In [ ]:
# ============================================================
# 21. Artifact Verification
# ============================================================

required_artifacts = [
    PREPROCESSOR_PATH,
    FEATURE_SCHEMA_PATH,

    PROCESSED_DIR / "X_train.npy",
    PROCESSED_DIR / "X_validation.npy",
    PROCESSED_DIR / "X_test.npy",
    PROCESSED_DIR / "y_train.npy",
    PROCESSED_DIR / "y_validation.npy",
    PROCESSED_DIR / "y_test.npy",

    INTERIM_DIR / "train_engineered.csv",
    INTERIM_DIR / "validation_engineered.csv",
    INTERIM_DIR / "test_engineered.csv",
]

artifact_status = []

for artifact in required_artifacts:
    exists = artifact.exists()
    artifact_status.append({
        "artifact": str(artifact.relative_to(PROJECT_ROOT)),
        "exists": exists,
        "size_bytes": artifact.stat().st_size if exists else 0
    })

artifact_status_df = pd.DataFrame(artifact_status)
display(artifact_status_df)

assert artifact_status_df["exists"].all(), (
    "One or more required artifacts are missing."
)

print("All required artifacts exist.")


# 17. Final Notebook 03 Verification

Notebook 03 is successful only when all gates below pass:

- [x] Raw dataset loaded.
- [x] Data quality checks passed.
- [x] Target separated before preprocessing.
- [x] Prediction-time feature policy documented.
- [x] `duration` excluded by default because of leakage concern.
- [x] Train/validation/test split is stratified.
- [x] Splits are mutually exclusive.
- [x] Feature engineering does not use the target.
- [x] Preprocessor fitted only on training data.
- [x] Validation/test transformed using the fitted training preprocessor.
- [x] No infinite processed values.
- [x] Exact transformed feature names recorded.
- [x] `preprocessor.joblib` saved.
- [x] `feature_schema.json` saved.
- [x] Train/validation/test arrays saved.
- [x] Engineered split data saved.
- [x] Saved preprocessor reloads successfully.
- [x] Reloaded transformation matches the original transformation.

## Expected Key Artifacts

```text
artifacts/
├── models/
│   └── preprocessor.joblib
│
└── metadata/
    └── feature_schema.json

data/
├── interim/
│   ├── train_engineered.csv
│   ├── validation_engineered.csv
│   └── test_engineered.csv
│
└── processed/
    ├── X_train.npy
    ├── X_validation.npy
    ├── X_test.npy
    ├── y_train.npy
    ├── y_validation.npy
    └── y_test.npy
```

**Do not proceed to Notebook 04 until these artifacts and the final verification output have been reviewed.**


In [ ]:
# ============================================================
# 22. FINAL AUTOMATED VERIFICATION
# ============================================================

assert df.shape == (45211, 17)
assert TARGET_COLUMN in df.columns

assert missing_cells == 0
assert duplicate_rows == 0
assert infinite_values == 0
assert not constant_columns

assert TARGET_COLUMN not in X_train.columns
assert TARGET_COLUMN not in X_val.columns
assert TARGET_COLUMN not in X_test.columns

assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_val_processed).all()
assert np.isfinite(X_test_processed).all()

assert PREPROCESSOR_PATH.exists()
assert FEATURE_SCHEMA_PATH.exists()

loaded_schema = json.loads(
    FEATURE_SCHEMA_PATH.read_text(encoding="utf-8")
)

assert loaded_schema["target_column"] == TARGET_COLUMN
assert loaded_schema["preprocessing"]["handle_unknown"] == "ignore"
assert loaded_schema["transformed_features"]["count"] == X_train_processed.shape[1]

print("=" * 80)
print("NOTEBOOK 03 VERIFICATION PASSED")
print("=" * 80)
print(f"Raw dataset shape          : {df.shape}")
print(f"Train shape                : {X_train.shape}")
print(f"Validation shape           : {X_val.shape}")
print(f"Test shape                 : {X_test.shape}")
print(f"Transformed feature count  : {X_train_processed.shape[1]}")
print(f"Excluded features          : {excluded_features}")
print(f"Engineered features        : {engineered_features}")
print(f"Preprocessor               : {PREPROCESSOR_PATH}")
print(f"Feature schema             : {FEATURE_SCHEMA_PATH}")
print("=" * 80)
print("Notebook 03 is complete. Review the outputs before proceeding to NEXT.")
